### Step 1: Use annotated max heights to estimate how repo height varies with repo area

In [1]:
import pandas as pd
import numpy as np
import os
import pyproj

In [2]:
# make save directory
out_dir = "../out/20240905/max_height_fit"
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

# load in flux file first
raw_data_root = "/Users/nick/Projects/gangaSandFlux/dat/20240905/"
filename = "StockpileFlux_Points_Final_corrected.csv"
sand_flux_df = pd.read_csv(os.path.join(raw_data_root, filename))
sand_flux_df.dropna(inplace=True)
sand_flux_df.reset_index(inplace=True, drop=True)

sand_flux_df.head(5)

,Stockpile_ID,AreaName,Date,numID,Type,PointName,POINT_X,POINT_Y
0,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414102,23.851631
1,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414108,23.851643
2,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414195,23.851627
3,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414383,23.851591
4,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414415,23.851577


In [3]:
# convert repo coordinates from lat lon to meters
projecter = pyproj.Proj(proj='utm', zone=45, ellps='WGS84', preserve_units=True)
point_tuple = projecter(sand_flux_df.loc[:, "POINT_X"], sand_flux_df.loc[:, "POINT_Y"])

# add to DF
sand_flux_df.loc[:, "point_x_m"] = point_tuple[0]-np.min(point_tuple[0])
sand_flux_df.loc[:, "point_y_m"] = point_tuple[1]-np.min(point_tuple[1])

sand_flux_df.loc[:, "AreaName"] = sand_flux_df.loc[:, "AreaName"].astype(str)
sand_flux_df.loc[:, "numID"] = sand_flux_df.loc[:, "numID"].astype(int)
sand_flux_df.loc[:, "Date"] = sand_flux_df.loc[:, "Date"].astype(str)
sand_flux_df.loc[:, "Type"] = sand_flux_df.loc[:, "Type"].astype(str)
# sand_flux_df = sand_flux_df.loc[sand_flux_df["Type"] != "nan"].reset_index(inplace=True, drop=True)


# make new ID variable
id_vec = [sand_flux_df.loc[p, "AreaName"] + "_" + sand_flux_df.loc[p, "Date"] + "_" 
                 + str(sand_flux_df.loc[p, "numID"]) for p in range(sand_flux_df.shape[0])] 
sand_flux_df.loc[:, "repoID"] = id_vec

In [4]:
def PolyArea(x,y):
    return 0.5*np.abs(np.dot(x,np.roll(y,1))-np.dot(y,np.roll(x,1)))

In [5]:
from tqdm import tqdm

# iterate through repos and calculate mh distances
repo_index = np.unique(sand_flux_df["repoID"])
mh_length_vec = []
repo_area_vec = []
point_name_vec = []
sub_ind_vec = []

for p, point_name in enumerate(tqdm(repo_index)):
    
    # filter
    df_temp = sand_flux_df.loc[sand_flux_df["repoID"]==point_name, :]
    df_temp = df_temp.reset_index(drop=True)
    
    # do we have mh info (only present if there is a plateau)?
    type_vec = df_temp["Type"].to_numpy()
    mh_flags = np.asarray([1 if "mh" in p else 0 for p in type_vec])
    base_flags = np.asarray([1 if "base" in p else 0 for p in type_vec])
    
    if np.any(mh_flags) and df_temp.shape[0] > 10 and np.any(base_flags):
        mh_index = np.unique(df_temp.loc[mh_flags==1, "Type"])
        n = len(mh_index) // 2
        
        for m, mh in enumerate(mh_index): # there can be multiple piles in the same repo
            mh_ft = np.where(type_vec==mh_index)[0]
            
            if len(mh_ft) == 2:
                # get length components
                dx = df_temp.loc[mh_ft[0], "point_x_m"] - df_temp.loc[mh_ft[1], "point_x_m"]
                dy = df_temp.loc[mh_ft[0], "point_y_m"] - df_temp.loc[mh_ft[1], "point_y_m"]
                
                # extract base points
                if n == 0:
                    key = "base"
                else:
                    key = "base" + str(m+1)
                    
                base_points_xy = df_temp.loc[type_vec==key, ["point_x_m", "point_y_m"]].to_numpy()
                assert base_points_xy.shape[0] > 0
                # calculate area
                area = PolyArea(base_points_xy[:, 0], base_points_xy[:, 1])
                
                # add info
                mh_length_vec.append(np.sqrt(dx**2 + dy**2))
                sub_ind_vec.append(m)
                point_name_vec.append(point_name)
                repo_area_vec.append(area)

  7%|▋         | 52/752 [00:00<00:01, 512.44it/s]/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_32628/3512608128.py:26: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  mh_ft = np.where(type_vec==mh_index)[0]
/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_32628/3512608128.py:26: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  mh_ft = np.where(type_vec==mh_index)[0]
/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_32628/3512608128.py:26: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  mh_ft = np.where(type_vec==mh_index)[0]
/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_32628/3512608128.py:26: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  mh_ft = np.where(type_vec==mh_index)[0]
 76%|███████▌  | 573/752 [00:01<00:00, 552.01it/s]/var/folders/m7/tpjttxb92tl8b4c9svgvg

#### Use bootstrap resampling to estimate the mean and standard error of the max height as a function of repo linear dimension

In [6]:
import scipy

repo_areas = np.asarray(repo_area_vec)
repo_heights = np.asarray(mh_length_vec) * np.tan(np.pi/6)
# fit height function
# [max_h, area_hm, n]
x0 = [9, 2e3, 1]
# max_h = np.percentile(repo_heights, 99)
param_fit_vec = []
n_boots = 100
boot_options = range(len(repo_areas))
repo_lengths = np.sqrt(repo_areas)

for n in tqdm(range(n_boots), "Performing bootstrap fits..."):
    boot_indices = np.random.choice(boot_options, len(boot_options), replace=True)
    
    def height_fun(params, repo_areas=repo_lengths[boot_indices]):
        h = np.divide(params[0] * repo_areas**params[2], (params[1]**params[2] + repo_areas**params[2]))
        return h

    def loss_fun(params, repo_heights=repo_heights[boot_indices]):
        h_hat = height_fun(params)
        return repo_heights - h_hat


    params_fit = scipy.optimize.least_squares(loss_fun, x0, bounds=[(0, 0, 0), (np.inf, np.inf, np.inf)])
    
    param_fit_vec.append(params_fit)
    

# get full fit on all paramters
def height_fun(params, repo_areas=repo_lengths):
        h = np.divide(params[0] * repo_areas**params[2], (params[1]**params[2] + repo_areas**params[2]))
        return h

def loss_fun(params, repo_heights=repo_heights):
    h_hat = height_fun(params)
    return repo_heights - h_hat

param_fit_full = scipy.optimize.least_squares(loss_fun, x0, bounds=[(0, 0, 0), (np.inf, np.inf, np.inf)])

Performing bootstrap fits...: 100%|██████████| 100/100 [00:01<00:00, 59.34it/s]


### Calculate the starndard error and mean as a function of length
Use this as a basis for estimating the standard deviation of the data about the inferred mean. We will use this for sampling

In [10]:
import plotly.graph_objects as go 
import plotly.express as px
from scipy.spatial import distance_matrix

length_vec = np.linspace(0, 500, 1000)
height_trend_array = np.empty((n_boots, len(length_vec)))

dists = distance_matrix(repo_lengths[:, np.newaxis], length_vec[:, np.newaxis])
l_indices = np.argmin(dists, axis=1)

for n in range(n_boots):
    height_trend_array[n, :] = height_fun(param_fit_vec[n].x, repo_areas=length_vec)

height_mean = height_fun(param_fit_vec[n].x, repo_areas=length_vec)
height_mean[0] = 0 # np.mean(height_trend_array, axis=0)
height_se = np.std(height_trend_array, axis=0)
ub = height_mean + height_se
lb = height_mean - height_se

# numerically estimage SD by expanding SE by constant until 68.27% of data fall within band
increment = 0.01
factor = 1
height_sigma = height_se
frac_inside = 0
while frac_inside < .6827:
    factor += increment
#     height_sigma *= factor
    ubs = height_mean + height_sigma*factor
    lbs = height_mean - height_sigma*factor
    frac_inside = np.mean((repo_heights <= ubs[l_indices]) & (repo_heights >= lbs[l_indices]))
    
fig = px.scatter(x=np.sqrt(repo_areas), y=repo_heights)
fig.add_trace(go.Scatter(x=length_vec, y=height_mean, mode="lines"))
fig.add_trace(go.Scatter(
        x=np.c_[length_vec, length_vec[::-1]].T.ravel(), 
        y=np.c_[ubs, lbs[::-1]].T.ravel(), # upper, then lower reversed
        fill='toself',
        fillcolor='rgba(0,0,0,0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    ))
fig.add_trace(go.Scatter(
        x=np.c_[length_vec, length_vec[::-1]].T.ravel(), 
        y=np.c_[ub, lb[::-1]].T.ravel(), # upper, then lower reversed
        fill='toself',
        fillcolor='rgba(0,0,0,0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    ))
fig.show()

### Save

In [9]:
import pickle

# Write the fit object to file
with open(os.path.join(out_dir, 'max_height_fit.pkl'), 'wb') as f:
    pickle.dump(param_fit_full, f)

In [15]:
import pandas as pd

# make lookup table
height_ref_df = pd.DataFrame(length_vec, columns=["side_dim_m"])
height_ref_df["max_height_m"] = height_mean
height_ref_df["height_se"] = height_se
height_ref_df["height_sd"] = height_se*factor
height_ref_df["overall_max_m"] = np.max(repo_heights)

height_ref_df.to_csv(os.path.join(out_dir, 'height_df.csv'), index=False)